In [21]:
import streamlit as st
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph,START,END 
from dotenv import load_dotenv
import os
from pydantic import BaseModel,Field
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [16]:
load_dotenv()


AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

In [17]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str
    

In [18]:
def generate_joke(state:JokeState):
    prompt = f"Generate a joke on the topic{state['topic']} "
    response = model.invoke(prompt).content
    return {
        'joke': response,
    }

In [19]:
def generate_explaination(state:JokeState):
    prompt = f"Explain the joke {state['joke']} in detail"
    response = model.invoke(prompt).content
    return {
        'explaination': response,
    }

In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'programming'}, config=config1)




{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.',
 'explaination': 'It’s a **pun** based on two meanings of **bugs** and a common idea about **light**.\n\n### The setup\n**“Why do programmers prefer dark mode?”**\n\nThis sounds like a normal tech joke, because programmers often do use dark mode in code editors and terminals.\n\n### The punchline\n**“Because light attracts bugs.”**\n\nThis works in two ways:\n\n1. **Literal meaning**\n   In real life, many insects are attracted to light.\n   So “light attracts bugs” means lamps and bright lights draw in actual bugs.\n\n2. **Programming meaning**\n   In software, a **bug** means an error or flaw in code.\n   So the joke pretends that using **light mode** would somehow attract software bugs.\n\n### Why it’s funny\nThe humor comes from:\n- **Wordplay**: “bugs” means both insects and coding errors.\n- **Absurd logic**: it jokingly treats programming bugs like real insects that fly towa

Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 4500
API Key: lsv2_********************************************1a
post: trace=e563dcf2-faa9-4d1c-b7c5-9e1106994c2b,id=e563dcf2-faa9-4d1c-b7c5-9e1106994c2b; trace=e563dcf2-faa9-4d1c-b7c5-9e1106994c2b,id=9d17fd56-ff60-4205-a405-67fb31804c9f; trace=e563dcf2-faa9-4d1c-b7c5-9e1106994c2b,id=c29a6a7b-f701-4324-b8eb-c21b8aea8127
Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/b

In [8]:
workflow.get_state(config1)


StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explaination': 'It’s a **pun** based on two meanings of **bugs** and a bit of programmer culture.\n\n### The setup\n**“Why do programmers prefer dark mode?”**\n\nThis sounds like a normal question about software developers liking dark-themed screens.\n\n### The punchline\n**“Because light attracts bugs.”**\n\nThis works because:\n\n1. **Literal meaning:**  \n   In real life, many insects (“bugs”) are attracted to light.\n\n2. **Programming meaning:**  \n   In software, a **bug** means an error or flaw in code.\n\nSo the joke pretends that if programmers use **light mode**, the “light” will attract more “bugs” into their programs.\n\n### Why it’s funny\n- It combines a **real-world fact** with a **technical term**.\n- It treats software bugs as if they were actual insects flying toward a lamp.\n- Programmers often joke about bugs constantly appearing in their co

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explaination': 'It’s a **pun** based on two meanings of **bugs** and a bit of programmer culture.\n\n### The setup\n**“Why do programmers prefer dark mode?”**\n\nThis sounds like a normal question about software developers liking dark-themed screens.\n\n### The punchline\n**“Because light attracts bugs.”**\n\nThis works because:\n\n1. **Literal meaning:**  \n   In real life, many insects (“bugs”) are attracted to light.\n\n2. **Programming meaning:**  \n   In software, a **bug** means an error or flaw in code.\n\nSo the joke pretends that if programmers use **light mode**, the “light” will attract more “bugs” into their programs.\n\n### Why it’s funny\n- It combines a **real-world fact** with a **technical term**.\n- It treats software bugs as if they were actual insects flying toward a lamp.\n- Programmers often joke about bugs constantly appearing in their c

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'sports'}, config=config2)

{'topic': 'sports',
 'joke': 'Why did the soccer ball get promoted?\n\nBecause it was always kicking goals at work.',
 'explaination': 'The joke is a **pun** based on the phrase **“kicking goals.”**\n\n### Why it’s funny\nIn normal workplace language, **“kicking goals”** means:\n- achieving targets\n- doing really well\n- succeeding at work\n\nSo if someone says:\n> “She’s really kicking goals at work,”\n\nthey mean she’s performing excellently.\n\n### The soccer connection\nA **soccer ball** is literally used to **kick goals** in soccer:\n- players kick the ball\n- the ball goes into the goal\n\nSo the joke takes the idiom **figuratively** and applies it **literally** to a soccer ball.\n\n### How the punchline works\n**Setup:**  \n> “Why did the soccer ball get promoted?”\n\nThis sounds like it’s going to be a workplace joke about good performance.\n\n**Punchline:**  \n> “Because it was always kicking goals at work.”\n\nThis works in two ways at once:\n1. **Figurative meaning:** it wa

Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 4485
API Key: lsv2_********************************************1a
post: trace=d7e8d523-8ec0-4041-b01b-2fb080d805ae,id=d7e8d523-8ec0-4041-b01b-2fb080d805ae; trace=d7e8d523-8ec0-4041-b01b-2fb080d805ae,id=0ff92239-6041-4684-8265-56e33d07b16d; trace=d7e8d523-8ec0-4041-b01b-2fb080d805ae,id=50918515-98b5-4119-b330-d9e55d69f371
Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/b

TIME TRAVEL DEBUGGING

In [12]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f131b58-dd93-62bb-8000-793b32f3b320"}})

StateSnapshot(values={'topic': 'programming'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f131b58-dd93-62bb-8000-793b32f3b320'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-06T12:38:46.385929+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f131b58-dd90-6b16-bfff-1297ce541d74'}}, tasks=(PregelTask(id='3c55eaaf-30f3-82a6-289d-b662a6c64b94', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.'}),), interrupts=())

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f131b58-dd93-62bb-8000-793b32f3b320"}})

{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.',
 'explaination': 'The joke is a **pun** based on two meanings of the word **“bugs.”**\n\n## The setup\n**“Why do programmers prefer dark mode?”**\n\nThis sounds like a normal question about software developers. “Dark mode” is a display setting where backgrounds are dark instead of bright white.\n\n## The punchline\n**“Because light attracts bugs.”**\n\nThis works because:\n\n### 1. In real life\nActual **bugs/insects** are often attracted to light sources, like lamps or porch lights.\n\n### 2. In programming\nA **bug** is also a word for an error or flaw in software.\n\nSo the joke pretends that if programmers use **light mode**, the “light” will attract more “bugs” in their code, like a lamp attracting insects.\n\n## Why it’s funny\nThe humor comes from:\n- **Wordplay**: “bugs” means both insects and software errors.\n- **Absurd literal logic**: It treats programming bugs as if th

Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 4656
API Key: lsv2_********************************************1a
post: trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=1f1ea3d0-c9fb-4913-b476-976780f94bc6; trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=d06af966-4ff4-43b9-a87e-a1a48b6eb17e; trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=8b4c1a2b-b1d8-462b-9b37-37793fbb6ad0


In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explaination': 'The joke is a **pun** based on two meanings of the word **“bugs.”**\n\n## The setup\n**“Why do programmers prefer dark mode?”**\n\nThis sounds like a normal question about software developers. “Dark mode” is a display setting where backgrounds are dark instead of bright white.\n\n## The punchline\n**“Because light attracts bugs.”**\n\nThis works because:\n\n### 1. In real life\nActual **bugs/insects** are often attracted to light sources, like lamps or porch lights.\n\n### 2. In programming\nA **bug** is also a word for an error or flaw in software.\n\nSo the joke pretends that if programmers use **light mode**, the “light” will attract more “bugs” in their code, like a lamp attracting insects.\n\n## Why it’s funny\nThe humor comes from:\n- **Wordplay**: “bugs” means both insects and software errors.\n- **Absurd literal logic**: It treats progr

Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 20037
API Key: lsv2_********************************************1a
post: trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=3aee736c-2ce7-4ca5-a57c-9950273f9ca9; trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=caaaf273-6336-40b9-9c98-f2f8e5366a6f; patch: trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=1f1ea3d0-c9fb-4913-b476-976780f94bc6; trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=d06af966-4ff4-43b9-a87e-a1a48b6eb17e; trace=1f1ea3d0-c9fb-4913-b476-976780f94bc6,id=8b4c1a2b-b